# QueryMate on Colab: Qwen2.5-Coder-3B Baseline vs QLoRA

This notebook follows your exact plan:
1. Evaluate **without fine-tuning** (baseline)
2. Fine-tune with **QLoRA** on a train split
3. Re-evaluate on the same held-out test split

It keeps your core project logic intact by reusing:
- RAG schema retrieval
- Agentic workflow (schema expert, planner, SQL coder, validator)
- CoT-style planning prompt
- Existing evaluator (execution accuracy)

## 1) Install dependencies
If you restart runtime, run this cell again.

In [ ]:
!pip -q install --upgrade pip

# Remove preinstalled telemetry packages that conflict with Chroma deps on Colab images
!pip -q uninstall -y google-adk opentelemetry-exporter-gcp-logging opentelemetry-exporter-otlp-proto-http opentelemetry-exporter-otlp-proto-grpc || true

!pip -q install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip -q install transformers datasets accelerate bitsandbytes peft trl sentence-transformers chromadb
!pip -q install langchain langchain-groq python-dotenv ollama

## 2) Clone your repo and move into project
Set `REPO_URL` to your QueryMate GitHub URL.

In [ ]:
import os
import shutil
import subprocess
import time
from pathlib import Path

REPO_URL = "https://github.com/Arul-7781/QueryMate.git"
REPO_BRANCH = "validation"
FALLBACK_BRANCH = "main"
REPO_DIR = Path('/content/QueryMate')

def run_cmd(cmd, check=True):
    print('$', ' '.join(cmd))
    res = subprocess.run(cmd, text=True, capture_output=True)
    if res.stdout.strip():
        print(res.stdout.strip())
    if res.returncode != 0:
        if res.stderr.strip():
            print(res.stderr.strip())
        if check:
            raise RuntimeError(f"Command failed ({res.returncode}): {' '.join(cmd)}")
    return res

# Optional auth for private repos: set GITHUB_TOKEN in Colab secrets/env
token = os.getenv('GITHUB_TOKEN', '').strip()
auth_url = REPO_URL.replace('https://', f'https://{token}@') if token else REPO_URL

def branch_exists(url, branch):
    res = subprocess.run(['git', 'ls-remote', '--heads', url, branch], text=True, capture_output=True)
    return res.returncode == 0 and bool(res.stdout.strip())

target_branch = REPO_BRANCH if branch_exists(auth_url, REPO_BRANCH) else FALLBACK_BRANCH
if target_branch != REPO_BRANCH:
    print(f"Warning: branch '{REPO_BRANCH}' not found. Falling back to '{FALLBACK_BRANCH}'.")

# Clean stale non-git directory from interrupted clone
if REPO_DIR.exists() and not (REPO_DIR / '.git').exists():
    print(f"Removing stale directory: {REPO_DIR}")
    shutil.rmtree(REPO_DIR)

if not REPO_DIR.exists():
    clone_cmd = ['git', 'clone', '--branch', target_branch, '--single-branch', auth_url, str(REPO_DIR)]
    last_error = None
    for attempt in range(1, 4):
        print(f"Clone attempt {attempt}/3")
        res = subprocess.run(clone_cmd, text=True, capture_output=True)
        if res.returncode == 0:
            if res.stdout.strip():
                print(res.stdout.strip())
            break
        last_error = res.stderr.strip() or res.stdout.strip()
        print(last_error)
        time.sleep(2 * attempt)
    else:
        raise RuntimeError(f"Clone failed after 3 attempts. Last error: {last_error}")
else:
    run_cmd(['git', '-C', str(REPO_DIR), 'fetch', 'origin', target_branch], check=False)
    run_cmd(['git', '-C', str(REPO_DIR), 'checkout', target_branch])
    run_cmd(['git', '-C', str(REPO_DIR), 'pull', 'origin', target_branch], check=False)

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

branch_res = subprocess.run(
    ['git', '-C', str(REPO_DIR), 'rev-parse', '--abbrev-ref', 'HEAD'],
    text=True, capture_output=True
 )
branch_name = branch_res.stdout.strip()
if not branch_name:
    # Fallback for detached HEAD / unusual git states
    detached_res = subprocess.run(
        ['git', '-C', str(REPO_DIR), 'describe', '--all', '--exact-match', 'HEAD'],
        text=True, capture_output=True
    )
    branch_name = detached_res.stdout.strip() or '(unknown/detached)'

print('Current branch:', branch_name)
if target_branch in ('validation', 'main') and branch_name not in (target_branch, f'refs/heads/{target_branch}') and not branch_name.endswith('/' + target_branch):
    print(f"Warning: expected branch '{target_branch}', but got '{branch_name}'.")

print('Top-level files:', sorted(os.listdir('.'))[:20])

## 3) Load real 200-case golden set and split fairly
This uses the 200 genuine SQL tasks in tests/golden_set.json (no paraphrase augmentation).

Fairness rule: we do stratified splitting by (difficulty, category) to preserve task-distribution across train/val/test.

In [ ]:
import json
import random
from collections import defaultdict
from pathlib import Path

random.seed(42)
TARGET_SIZE = 200
GOLDEN_PATH = Path('tests/golden_set.json')

with open(GOLDEN_PATH, 'r') as f:
    all_cases = json.load(f)

all_cases = sorted(all_cases, key=lambda x: x['id'])
if len(all_cases) < TARGET_SIZE:
    raise ValueError(f"Expected at least {TARGET_SIZE} cases, found {len(all_cases)}")

# Use the first 200 by ID for reproducibility
cases = all_cases[:TARGET_SIZE]


def split_counts(n: int):
    if n <= 1:
        return n, 0, 0
    if n == 2:
        return 1, 0, 1

    n_train = int(0.70 * n)
    n_val = int(0.15 * n)
    n_test = n - n_train - n_val

    if n_train == 0:
        n_train = 1
    if n_val == 0:
        n_val = 1
    n_test = n - n_train - n_val

    if n_test <= 0:
        if n_train >= n_val and n_train > 1:
            n_train -= 1
        elif n_val > 1:
            n_val -= 1
        n_test = n - n_train - n_val

    return n_train, n_val, n_test


# Stratify by (difficulty, category)
groups = defaultdict(list)
for item in cases:
    key = (item.get('difficulty', 'unknown'), item.get('category', 'unknown'))
    groups[key].append(item)

train_cases, val_cases, test_cases = [], [], []
for key, items in groups.items():
    random.shuffle(items)
    n_train, n_val, n_test = split_counts(len(items))
    train_cases.extend(items[:n_train])
    val_cases.extend(items[n_train:n_train + n_val])
    test_cases.extend(items[n_train + n_val:n_train + n_val + n_test])

random.shuffle(train_cases)
random.shuffle(val_cases)
random.shuffle(test_cases)

print(f"Loaded cases: {len(cases)}")
print(f"Split sizes => Train: {len(train_cases)}, Val: {len(val_cases)}, Test: {len(test_cases)}")
print(f"Unique categories => {len(set(c['category'] for c in cases))}")

# Persist split files for reproducibility
Path('tests').mkdir(parents=True, exist_ok=True)
with open('tests/golden_set_200.json', 'w') as f:
    json.dump(cases, f, indent=2)
with open('tests/golden_set_200_train.json', 'w') as f:
    json.dump(train_cases, f, indent=2)
with open('tests/golden_set_200_val.json', 'w') as f:
    json.dump(val_cases, f, indent=2)
with open('tests/golden_set_200_test.json', 'w') as f:
    json.dump(test_cases, f, indent=2)

print('Saved split files under tests/')

## 4) Model runtime setup (HF for training, optional Ollama for baseline inference)
Important clarification:
- `from ollama import chat` is the correct API for inference with an Ollama-served model.
- QLoRA fine-tuning is **not** done through `ollama.chat`; it is done with HF/PEFT (as below).

This notebook therefore:
- uses HF/PEFT for training + fine-tuned evaluation,
- optionally supports Ollama `chat()` for baseline-only inference if you enable it.

In [ ]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

import gc
import torch
from dataclasses import dataclass
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL_ID = 'Qwen/Qwen2.5-Coder-3B-Instruct'
BASELINE_FALLBACK_MODEL_ID = 'Qwen/Qwen2.5-Coder-3B-Instruct'  # keep same model for compatibility
ALLOW_BASELINE_FALLBACK = False

OLLAMA_MODEL = 'qwen2.5-coder:3b'  # if using ollama baseline
USE_OLLAMA_BASELINE = False       # set True only if Ollama server/model is available

@dataclass
class LocalResponse:
    content: str

class HFRuntime:
    def __init__(self, model_id, adapter_path=None, max_new_tokens=128, max_memory=None):
        self.model_id = model_id
        self.max_new_tokens = max_new_tokens
        self.max_memory = max_memory or {0: '12GiB', 'cpu': '48GiB'}

        if torch.cuda.is_available():
            gc.collect()
            torch.cuda.empty_cache()

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True
        )

        self.tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        base_model = AutoModelForCausalLM.from_pretrained(
            model_id,
            quantization_config=bnb_config,
            device_map='auto',
            max_memory=self.max_memory,
            offload_folder='offload',
            offload_state_dict=True,
            low_cpu_mem_usage=True,
            trust_remote_code=True
        )

        if adapter_path is not None:
            self.model = PeftModel.from_pretrained(base_model, adapter_path)
        else:
            self.model = base_model

        # Reduce noisy generation warnings for greedy decoding
        if hasattr(self.model, 'generation_config') and self.model.generation_config is not None:
            for attr in ('temperature', 'top_p', 'top_k'):
                if hasattr(self.model.generation_config, attr):
                    setattr(self.model.generation_config, attr, None)

        self.model.eval()

    def _generate_once(self, prompt, temperature=0.0, use_chat_template=True):
        do_sample = temperature is not None and temperature > 0

        if use_chat_template:
            messages = [
                {
                    'role': 'system',
                    'content': 'You are a precise SQLite assistant. Return only SQL when SQL is requested.'
                },
                {'role': 'user', 'content': prompt},
            ]
            input_text = self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
        else:
            input_text = prompt

        inputs = self.tokenizer(input_text, return_tensors='pt')
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        inputs = {k: v.to(device) for k, v in inputs.items()}

        # Use max_length (not max_new_tokens) to avoid max_length/max_new_tokens warning
        max_length = int(inputs['input_ids'].shape[-1] + self.max_new_tokens)

        gen_kwargs = {
            'max_length': max_length,
            'do_sample': do_sample,
            'pad_token_id': self.tokenizer.eos_token_id,
            'eos_token_id': self.tokenizer.eos_token_id,
        }

        if do_sample:
            gen_kwargs['temperature'] = max(float(temperature), 0.01)
            gen_kwargs['top_p'] = 0.95

        with torch.no_grad():
            output_ids = self.model.generate(**inputs, **gen_kwargs)

        new_tokens = output_ids[0][inputs['input_ids'].shape[-1]:]
        out = self.tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        return out

    def invoke(self, prompt, temperature=0.0):
        # First pass with chat template (best for instruct models like Qwen2.5-Coder)
        out = self._generate_once(prompt, temperature=temperature, use_chat_template=True)

        # Fallback if model returns empty output on first pass
        if not out:
            out = self._generate_once(prompt, temperature=temperature, use_chat_template=False)

        return LocalResponse(content=out.strip())

class OllamaRuntime:
    def __init__(self, model_name='qwen2.5-coder:3b'):
        self.model_name = model_name

    def invoke(self, prompt, temperature=0.0):
        from ollama import chat
        resp = chat(
            model=self.model_name,
            messages=[{'role': 'user', 'content': prompt}],
            options={'temperature': float(temperature)}
        )
        return LocalResponse(content=resp['message']['content'].strip())

def patch_agents_with_runtime(runtime):
    import src.agents as agents

    # Store runtime on module so we can explicitly clear it later.
    agents._ACTIVE_RUNTIME = runtime

    def local_invoke_with_fallback(prompt, temperature=0.0):
        active_runtime = getattr(agents, '_ACTIVE_RUNTIME', None)
        if active_runtime is None:
            raise RuntimeError('No active runtime patched. Call patch_agents_with_runtime(runtime) first.')
        return active_runtime.invoke(prompt, temperature=temperature)

    # Keep all agentic logic unchanged; only swap model invocation backend
    agents.invoke_with_fallback = local_invoke_with_fallback
    return agents


def clear_runtime_patch():
    import src.agents as agents
    if hasattr(agents, '_ACTIVE_RUNTIME'):
        agents._ACTIVE_RUNTIME = None

print('Runtime classes ready. USE_OLLAMA_BASELINE =', USE_OLLAMA_BASELINE)

### Optional: Start Ollama in Colab (only if `USE_OLLAMA_BASELINE = True`)
Run the next cell once to install/start Ollama and pull `qwen2.5-coder:3b` for baseline inference via `ollama.chat(...)`.

In [ ]:
# Optional Ollama bootstrap for Colab baseline
import os
import subprocess

if USE_OLLAMA_BASELINE:
    print('Installing and starting Ollama...')
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
    subprocess.Popen("ollama serve", shell=True)
    os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'

    print('Pulling model:', OLLAMA_MODEL)
    subprocess.run(f"ollama pull {OLLAMA_MODEL}", shell=True, check=True)
    print('Ollama ready.')
else:
    print('USE_OLLAMA_BASELINE=False; skipping Ollama bootstrap.')

## 5) Baseline evaluation (no fine-tuning)

In [ ]:
import gc
import sys
import importlib.util
from pathlib import Path

REPO_ROOT = Path(str(REPO_DIR)) if 'REPO_DIR' in globals() else Path('/content/QueryMate')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

try:
    import tests.evaluator as evaluator
except ModuleNotFoundError:
    eval_path = REPO_ROOT / 'tests' / 'evaluator.py'
    spec = importlib.util.spec_from_file_location('querymate_evaluator', eval_path)
    evaluator = importlib.util.module_from_spec(spec)
    if spec is None or spec.loader is None:
        raise RuntimeError(f'Could not load evaluator from {eval_path}')
    spec.loader.exec_module(evaluator)

# Optional: speed up local runs (no API rate limits locally)
evaluator.SLEEP_BETWEEN = 0.0


def is_oom_error(exc):
    txt = str(exc).lower()
    name = exc.__class__.__name__.lower()
    return (
        'outofmemoryerror' in name
        or 'cuda out of memory' in txt
        or 'out of memory' in txt
        or 'cudnn_status_alloc_failed' in txt
    )


# Ensure previous runtime references are released before (re)loading baseline model.
if 'clear_runtime_patch' in globals():
    clear_runtime_patch()
if 'baseline_runtime' in globals():
    try:
        del baseline_runtime
    except Exception:
        pass
if torch.cuda.is_available():
    gc.collect()
    torch.cuda.empty_cache()

if USE_OLLAMA_BASELINE:
    print('Using Ollama baseline with model:', OLLAMA_MODEL)
    baseline_runtime = OllamaRuntime(OLLAMA_MODEL)
    BASELINE_MODEL_IN_USE = OLLAMA_MODEL
else:
    print('Using HF baseline with model:', BASE_MODEL_ID)

    try:
        baseline_runtime = HFRuntime(
            BASE_MODEL_ID,
            max_new_tokens=96,
            max_memory={0: '10GiB', 'cpu': '64GiB'}
        )
        BASELINE_MODEL_IN_USE = BASE_MODEL_ID
    except Exception as exc:
        if ALLOW_BASELINE_FALLBACK and is_oom_error(exc):
            print('OOM while loading baseline model. Falling back to:', BASELINE_FALLBACK_MODEL_ID)
            if torch.cuda.is_available():
                gc.collect()
                torch.cuda.empty_cache()
            baseline_runtime = HFRuntime(
                BASELINE_FALLBACK_MODEL_ID,
                max_new_tokens=96,
                max_memory={0: '10GiB', 'cpu': '64GiB'}
            )
            BASELINE_MODEL_IN_USE = BASELINE_FALLBACK_MODEL_ID
        else:
            raise

print('Baseline model in use:', BASELINE_MODEL_IN_USE)
patch_agents_with_runtime(baseline_runtime)

# Set to 5 for a quick smoke test, then set to None for full test split
BASELINE_LIMIT = 5
cases_to_run = test_cases[:BASELINE_LIMIT] if BASELINE_LIMIT else test_cases
print(f'Running baseline on {len(cases_to_run)} test cases')

baseline_report = evaluator.run_evaluation(cases_to_run)
print('Baseline Execution Accuracy:', baseline_report['execution_accuracy'])

# Free memory before QLoRA training (HF runtime only)
if not USE_OLLAMA_BASELINE:
    clear_runtime_patch()
    del baseline_runtime
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 6) QLoRA fine-tuning setup
We train only LoRA adapters (not full model weights) for T4 feasibility.

In [ ]:
import sqlite3
import inspect
from datasets import Dataset
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

DB_PATH = 'data/company.db'

def get_schema_text(db_path=DB_PATH):
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute("SELECT name, sql FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name;")
    rows = cur.fetchall()
    conn.close()
    return '\n\n'.join([f"{name}: {sql}" for name, sql in rows if sql])

schema_text = get_schema_text()

def build_text(example):
    prompt = (
        "You are an expert SQLite Developer. Generate SQL for the question.\n"
        "Return only SQL, no markdown, no explanation.\n\n"
        f"Question: {example['question']}\n"
        f"Schema:\n{schema_text}\n\n"
        "SQL:\n"
    )
    return prompt + example['sql']

train_ds = Dataset.from_list([{'text': build_text(x)} for x in train_cases])
val_ds = Dataset.from_list([{'text': build_text(x)} for x in val_cases])

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

ft_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
if ft_tokenizer.pad_token is None:
    ft_tokenizer.pad_token = ft_tokenizer.eos_token

def tokenize_batch(batch):
    return ft_tokenizer(batch['text'], truncation=True, max_length=1024)

train_tok = train_ds.map(tokenize_batch, batched=True, remove_columns=['text'])
val_tok = val_ds.map(tokenize_batch, batched=True, remove_columns=['text'])

ft_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    max_memory={0: '10GiB', 'cpu': '64GiB'},
    offload_folder='offload',
    offload_state_dict=True,
    low_cpu_mem_usage=True,
    trust_remote_code=True
)
ft_model.config.use_cache = False
ft_model.gradient_checkpointing_enable()
ft_model = prepare_model_for_kbit_training(ft_model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
)

ft_model = get_peft_model(ft_model, lora_config)
ft_model.print_trainable_parameters()

collator = DataCollatorForLanguageModeling(tokenizer=ft_tokenizer, mlm=False)

training_kwargs = {
    'output_dir': 'outputs/qwen25coder3b-qlora',
    'per_device_train_batch_size': 1,
    'per_device_eval_batch_size': 1,
    'gradient_accumulation_steps': 8,
    'learning_rate': 2e-4,
    'num_train_epochs': 2,
    'fp16': True,
    'gradient_checkpointing': True,
    'optim': 'paged_adamw_8bit',
    'save_strategy': 'epoch',
    'logging_steps': 5,
    'report_to': 'none',
}

# Transformers API compatibility: some versions use eval_strategy,
# older versions use evaluation_strategy.
ta_params = inspect.signature(TrainingArguments.__init__).parameters
if 'evaluation_strategy' in ta_params:
    training_kwargs['evaluation_strategy'] = 'epoch'
else:
    training_kwargs['eval_strategy'] = 'epoch'

args = TrainingArguments(**training_kwargs)

trainer = Trainer(
    model=ft_model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=collator
)

trainer.train()

ADAPTER_DIR = 'outputs/qwen25coder3b-qlora/final_adapter'
trainer.model.save_pretrained(ADAPTER_DIR)
ft_tokenizer.save_pretrained(ADAPTER_DIR)
print('Saved adapter to:', ADAPTER_DIR)

## 7) Evaluation after QLoRA fine-tuning

In [ ]:
import gc

del ft_model
gc.collect()
torch.cuda.empty_cache()

# Fine-tuned evaluation always uses HF runtime with LoRA adapter
ft_runtime = HFRuntime(
    BASE_MODEL_ID,
    adapter_path=ADAPTER_DIR,
    max_new_tokens=96,
    max_memory={0: '10GiB', 'cpu': '64GiB'}
)
patch_agents_with_runtime(ft_runtime)

finetuned_report = evaluator.run_evaluation(test_cases)
print('Fine-tuned Execution Accuracy:', finetuned_report['execution_accuracy'])

## 8) Compare baseline vs fine-tuned

In [ ]:
import pandas as pd

def effective_denominator(report):
    return max(report['total'] - report['errors'], 1)

summary = pd.DataFrame([
    {
        'Run': 'Baseline (No FT)',
        'Passed': baseline_report['passed'],
        'Failed': baseline_report['failed'],
        'Errors': baseline_report['errors'],
        'Execution Accuracy': baseline_report['passed'] / effective_denominator(baseline_report)
    },
    {
        'Run': 'QLoRA Fine-tuned',
        'Passed': finetuned_report['passed'],
        'Failed': finetuned_report['failed'],
        'Errors': finetuned_report['errors'],
        'Execution Accuracy': finetuned_report['passed'] / effective_denominator(finetuned_report)
    }
])

display(summary)

## Notes
- This notebook keeps your QueryMate orchestration logic intact and only swaps the model backend.
- If Colab OOM occurs, reduce max sequence length (1024 -> 768), LoRA rank (16 -> 8), or epochs (2 -> 1).
- For stronger fine-tuning conclusions, expand training data beyond the current golden set.